<a href="https://colab.research.google.com/github/niteshgee123-droid/colab/blob/main/Rainfall_based_flood_impact_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nepal ERTF Flood Alert System
Complete pipeline: train on historical data, then predict flood risk for new forecasts

## Setup & Initialize

In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
import time

ee.Authenticate()
ee.Initialize(project='ee-niteshgee123')

Define study area (Nepal), data sources, and output folder

In [ ]:
nepal = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017').filter(ee.Filter.eq('country_na', 'Nepal'))
aoi = nepal.geometry()

asset_base = 'projects/ee-niteshgee123/assets/npl_ertf'
months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

train_start = 1990
train_end = 2021
cell_size = 0.15
durations = [1, 2, 3]

chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
gsw_occ = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
gsw_month = ee.ImageCollection('JRC/GSW1_4/MonthlyHistory')
perm_water = gsw_occ.gt(90)

# STAGE 1: TRAINING
Train the model on 30 years of historical rainfall and flood data
This computes percentiles and logistic regression coefficients, saves them to Google

## Helper functions for rainfall and flood data

In [ ]:
def rainfall_sum(start_date, days):
    return chirps.filterDate(start_date, start_date.advance(days, 'day')).select('precipitation').sum()

def get_rain_samples(month, days):
    imgs = []
    for year in range(train_start, train_end + 1):
        month_start = ee.Date.fromYMD(year, month, 1)
        for w in range(min(5, max(1, 28 // days))):
            start = month_start.advance(w * days, 'day')
            imgs.append(rainfall_sum(start, days).rename('R').clip(aoi))
    return ee.ImageCollection(imgs)

## Part 1a: Compute and save percentile thresholds (P50, P80, P90, P96)
For each month and accumulation duration, get 30 years of historical rainfall windows
Then compute 50th, 80th, 90th, 96th percentile values per pixel across Nepal

In [ ]:
def save_thresholds(month, days):
    samples = get_rain_samples(month, days)
    pct = samples.reduce(ee.Reducer.percentile([50, 80, 90, 96]))
    m = months[month - 1]

    for img, pname in [(pct.select('R_p50'), 'p50'), (pct.select('R_p80'), 'p80'),
                        (pct.select('R_p90'), 'p90'), (pct.select('R_p96'), 'p96')]:
        asset_id = f'{asset_base}/npl_{pname}_{m}'
        task = ee.batch.Export.image.toAsset(
            image=img,
            description=f'npl_{pname}_{m}',
            assetId=asset_id,
            region=aoi,
            scale=5000,
            crs='EPSG:4326'
        )
        task.start()
        print(f'queued: {asset_id}')

Run threshold computation for all 12 months and 3 durations
This queues 144 export tasks to Google (12 months × 3 durations × 4 percentiles)

In [ ]:
for month in range(1, 13):
    for days in durations:
        save_thresholds(month, days)

time.sleep(60)

## Part 1b: Fit logistic regression coefficients (Slope S, Intercept I)
For each month + duration:
- Extract paired samples: rainfall amount + did flooding occur?
- Fit logistic regression at ~16km grid cell resolution (coarse enough for stable fits)
- Extract slope (S) and intercept (I) coefficients
- Save as spatial rasters

In [ ]:
def get_bounds():
    coords = aoi.bounds().coordinates().get(0).getInfo()
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return min(lons), min(lats), max(lons), max(lats)

def grid(bounds, size):
    lon_min, lat_min, lon_max, lat_max = bounds
    feats = []
    for i, lat in enumerate(np.arange(lat_min, lat_max, size)):
        for j, lon in enumerate(np.arange(lon_min, lon_max, size)):
            c_lon, c_lat = lon + size / 2, lat + size / 2
            feats.append(ee.Feature(ee.Geometry.Point([c_lon, c_lat]),
                                     {'row': i, 'col': j, 'lon': c_lon, 'lat': c_lat}))
    return ee.FeatureCollection(feats)

def fc_to_df(fc):
    return pd.DataFrame([f['properties'] for f in fc.getInfo()['features']])

In [ ]:
def flood_img(month, year):
    gsw = gsw_month.filter(ee.Filter.eq('year', year)).filter(ee.Filter.eq('month', month)).first()
    water = ee.Image(gsw).select('water')
    return water.eq(2).updateMask(perm_water.Not()).rename('flood')

def extract_samples(month, days, grid_fc):
    bands = []
    for year in range(train_start, train_end + 1):
        month_start = ee.Date.fromYMD(year, month, 1)
        bands.append(flood_img(month, year).rename(f'flood_{year}'))
        for w in range(min(5, max(1, 28 // days))):
            start = month_start.advance(w * days, 'day')
            bands.append(rainfall_sum(start, days).rename(f'R_{year}_{w}'))

    stacked = ee.Image.cat(bands)
    return fc_to_df(stacked.sampleRegions(collection=grid_fc, scale=5000, geometries=False))

In [ ]:
def to_long(df):
    parts = []
    for year in range(train_start, train_end + 1):
        fcol = f'flood_{year}'
        if fcol not in df.columns:
            continue
        for w in range(min(5, max(1, 28 // 2))):
            rcol = f'R_{year}_{w}'
            if rcol not in df.columns:
                continue
            sub = df[['row', 'col', 'lon', 'lat', rcol, fcol]].rename(columns={rcol: 'R', fcol: 'flood'})
            sub = sub.dropna(subset=['R', 'flood'])
            if len(sub) > 0:
                parts.append(sub)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def fit_cells(df):
    results = []
    for (row, col), g in df.groupby(['row', 'col']):
        if len(g) < 10 or g['flood'].nunique() < 2:
            continue
        try:
            model = LogisticRegression(penalty='l2', C=1.0, max_iter=1000)
            model.fit(g[['R']].values, g['flood'].values.astype(int))
            results.append({'lon': g['lon'].iloc[0], 'lat': g['lat'].iloc[0],
                             'S': float(model.coef_[0][0]), 'I': float(model.intercept_[0])})
        except:
            pass
    return results

In [ ]:
def save_logistic(month, days):
    print(f'training month {month}, {days}d...')
    bounds = get_bounds()
    grid_fc = grid(bounds, cell_size)

    raw = extract_samples(month, days, grid_fc)
    df = to_long(raw)

    if len(df) == 0:
        print(f'  skip: no data')
        return

    fitted = fit_cells(df)
    if not fitted:
        print(f'  skip: no valid cells')
        return

    feats = [ee.Feature(ee.Geometry.Point([r['lon'], r['lat']]), {'S': r['S'], 'I': r['I']}) for r in fitted]
    fc = ee.FeatureCollection(feats)

    scale_m = cell_size * 111000
    S = fc.reduceToImage(['S'], ee.Reducer.mean()).reproject(crs='EPSG:4326', scale=scale_m).rename('S')
    I = fc.reduceToImage(['I'], ee.Reducer.mean()).reproject(crs='EPSG:4326', scale=scale_m).rename('I')

    m = months[month - 1]
    for img, cname in [(S, 'slope'), (I, 'intercept')]:
        asset_id = f'{asset_base}/npl_cli_day{days}_{m}_{cname}_imerg'
        task = ee.batch.Export.image.toAsset(
            image=img.clip(aoi),
            description=f'npl_day{days}_{m}_{cname}',
            assetId=asset_id,
            region=aoi,
            scale=5000,
            crs='EPSG:4326'
        )
        task.start()
        print(f'  queued: {asset_id}')

Run logistic regression training for all 12 months and 3 durations
This queues 72 more export tasks (12 months × 3 durations × 2 coefficients: slope + intercept)

In [ ]:
for month in range(1, 13):
    for days in durations:
        save_logistic(month, days)

print('done, check assets in 10-15 min')

---
# STAGE 2: PREDICTION
Load the pre-trained static assets and run flood alert for a new forecast date
Change forecast_date and num_days below, then run cells

Set the forecast parameters

In [ ]:
forecast_date = '2026-08-19'
num_days = 2
source = 'GFS'

Get forecast rainfall from GFS or ECMWF

In [ ]:
def get_rain(start_date, days, src):
    end_date = start_date.advance(days, 'day')
    if src == 'GFS':
        col = ee.ImageCollection('NOAA/GFS0P25').filterDate(start_date, end_date).select('total_precipitation_surface')
    elif src == 'ECMWF':
        col = ee.ImageCollection('ECMWF/NRT_FORECAST/IFS/OPER').filterDate(start_date, end_date).select('total_precipitation_6hr')
    return col.sum().rename('R').clip(aoi)

start = ee.Date(forecast_date)
R = get_rain(start, num_days, source)
month_idx = int(start.get('month').getInfo()) - 1

Load pre-trained thresholds and coefficients from Google Earth Engine assets

In [ ]:
def get_p(month_idx):
    m = months[month_idx]
    return {
        'p50': ee.Image(asset_base + f'/npl_p50_{m}'),
        'p80': ee.Image(asset_base + f'/npl_p80_{m}'),
        'p90': ee.Image(asset_base + f'/npl_p90_{m}'),
        'p96': ee.Image(asset_base + f'/npl_p96_{m}'),
    }

def get_s(month_idx, days):
    m = months[month_idx]
    return ee.Image(asset_base + f'/npl_cli_day{days}_{m}_slope_imerg').rename('S')

def get_i(month_idx, days):
    m = months[month_idx]
    return ee.Image(asset_base + f'/npl_cli_day{days}_{m}_intercept_imerg').rename('I')

## Impact Score (X: 0-4)
Compare forecast rainfall against historical percentile thresholds
- X=0: below P50 (below median)
- X=1: P50-P80 (slightly above average)
- X=2: P80-P90 (quite high)
- X=3: P90-P96 (very high)
- X=4: above P96 (extremely rare)

In [ ]:
t = get_p(month_idx)
X = (ee.Image(0)
     .where(R.gte(t['p50']), 1)
     .where(R.gte(t['p80']), 2)
     .where(R.gte(t['p90']), 3)
     .where(R.gte(t['p96']), 4)
     .rename('X')
     .clip(aoi))

## Flood Likelihood Score (Y: 0-2)
Use logistic regression to estimate probability of flooding given the rainfall
V = 1 / (1 + e^-(S*R + I))  ... probability 0-1
- Y=0: V < 0.33 (low likelihood)
- Y=1: 0.33 <= V < 0.66 (moderate)
- Y=2: V >= 0.66 (high likelihood)

In [ ]:
S = get_s(month_idx, num_days)
I = get_i(month_idx, num_days)

V = ee.Image(1).divide(ee.Image(1).add(R.multiply(S).add(I).multiply(-1).exp())).rename('V')
Y = ee.Image(0).where(V.gte(0.33), 1).where(V.gte(0.66), 2).rename('Y')

## Final Composite Score (Z: 1-15) and Alert Classification
Combine Impact (X) and Likelihood (Y) with weighting
Z = 3*X + Y + 1

- Z < 4: Green (No Alert)
- 4 <= Z < 7: Yellow (Low Risk)
- 7 <= Z < 10: Orange (Moderate Risk)
- Z >= 10: Red (High Risk)

In [ ]:
Z = X.multiply(3).add(Y).add(1).rename('Z')
alert = (ee.Image(1)
         .where(Z.gte(4), 2)
         .where(Z.gte(7), 3)
         .where(Z.gte(10), 4)
         .rename('alertClass')
         .clip(aoi))

result = R.rename('R').addBands(X).addBands(V).addBands(Y).addBands(Z).addBands(alert)

## Display Flood Risk Map
Show Nepal with color-coded flood risk
Green = safe, Red = high risk

In [ ]:
palette = ['1a9850', 'ffff00', 'ff7f00', 'd7191c']

Map = geemap.Map()
Map.centerObject(nepal, 6)
Map.addLayer(nepal.style(color='black', fillColor='00000000'), {}, 'Nepal')
Map.addLayer(result.select('alertClass'), {'min': 1, 'max': 4, 'palette': palette}, 'ERTF')
Map.addLayer(result.select('R'), {'min': 0, 'max': 200, 'palette': ['ffffff', '4575b4', '313695']}, 'Rainfall (mm)', False)
Map.addLayer(result.select('Z'), {'min': 1, 'max': 15, 'palette': palette}, 'Score Z', False)

Map.add_legend(title='Alert', legend_dict={
    'No Alert': '1a9850',
    'Low': 'ffff00',
    'Moderate': 'ff7f00',
    'High': 'd7191c',
})

Map.show()